# Rulebook question answering

A TF-IDF retriever selects passages for multiple-choice questions. The generator receives those passages alongside the answer options and can abstain when the context is insufficient.

In [ ]:
import json
from pathlib import Path
import requests
from nlp_portfolio.rag import build_sparse_index, retrieve, format_mcq, option_id

BASE = 'https://raw.githubusercontent.com/ferrazzipietro/dataset-nlp-math/main/data_rag/excercise/'
folder = Path('data')
folder.mkdir(exist_ok=True)
for filename in ('rules.json', 'questions_mcq_student.json'):
    target = folder / filename
    if not target.exists():
        response = requests.get(BASE + filename, timeout=30)
        response.raise_for_status()
        target.write_bytes(response.content)
rules = json.loads((folder / 'rules.json').read_text())
questions = json.loads((folder / 'questions_mcq_student.json').read_text())
index = build_sparse_index(rules)
print('Rules:', len(rules), 'Questions:', len(questions))

In [ ]:
question = questions[0]
options = '\n'.join(f"{o['id']}. {o['text']}" for o in question['options'])
passages = retrieve(question['question'] + '\n' + options, index, top_k=5)
prompt = format_mcq(question['question'], question['options'], passages)
print('Retrieved IDs:', [item['id'] for item in passages])
print(prompt[:1600])

## Generating an answer

The answer extractor accepts one unambiguous option letter. Empty or malformed replies stay unanswered.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GENERATOR = 'Qwen/Qwen2.5-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained(GENERATOR)
mdl = AutoModelForCausalLM.from_pretrained(GENERATOR, device_map='auto',
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
results = []
for q in questions:
    option_text = '\n'.join(f"{opt['id']}. {opt['text']}" for opt in q['options'])
    chunks = retrieve(q['question'] + '\n' + option_text, index, top_k=5)
    message = format_mcq(q['question'], q['options'], chunks)
    inputs = tok.apply_chat_template([{'role':'user','content':message}],
        tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt').to(mdl.device)
    with torch.inference_mode():
        generated = mdl.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
    answer = tok.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    results.append({'id': q['id'], 'prediction': option_id(answer),
                    'answer': answer, 'evidence': [p['id'] for p in chunks]})
print('Answered:', len(results), 'Abstained:', sum(r['prediction'] is None for r in results))

In [ ]:
import pandas as pd
output = pd.DataFrame(results)
output[['id', 'prediction']].to_csv('rulebook_predictions.csv', index=False)
print(output.head())

Retrieval performance requires relevance annotations, while answer accuracy requires a trusted answer key. The available questions alone are insufficient for either score, so this notebook does not report a guessed accuracy.